# Web Scrapper Simple + OPENAIAPI

In [1]:
#Imports

import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI

In [3]:
# cargando Variables de Fichero .env

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

# checando la variable

if not api_key:
    print("No se encontro ninguna api ")
elif not api_key.startswith("sk-proj-"):
    print("Se encontró un aclave Api pero no es el fromato correcto")
elif api_key.strip() != api_key:
    print("Se encontro una api key pero parece tener espacios")
else:
    print("Se encontro la clave API y hasta ahora parece buena")

Se encontro la clave API y hasta ahora parece buena


In [4]:
openai = OpenAI()

In [5]:
class Website:
    """
    Una clase de utilidad para representar un sitio web que hemos scrappeado
    """

    def __init__(self, url):
        """
        Crea este objeto de sitio web a partir de la URL indicada utilizando la biblioteca BeautifulSoup
        """
        self.url = url
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No tiene título"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [7]:
pagina = Website("https://wired.com")
print(pagina.title)
print(pagina.text)

WIRED - The Latest in Technology, Science, Culture and Business | WIRED
Skip to main content
THE WIRED APP IS HERE
DOWNLOAD NOW »
Menu
WIRED
SECURITY
POLITICS
THE BIG STORY
BUSINESS
SCIENCE
CULTURE
REVIEWS
Menu
WIRED
Account
Account
Newsletters
Security
Politics
The Big Story
Business
Science
Culture
Reviews
Chevron
More
Expand
The Big Interview
Magazine
Events
WIRED Insider
WIRED Consulting
Newsletters
Podcasts
Video
Livestreams
WIRED Store
Search
Search
WIRED
Today’s Picks
Curating...
Headphone
Listen
Power Play
The Data Center Backlash Should Also Be a Climate Reckoning. It Isn’t Yet
Climate groups are latching onto the data center opposition movement, but it remains to be seen if they can get people to care about the facilities’ impact on the atmosphere.
By
Molly Taft
Fuzzy Logic
Meta’s Muse Is Adults-Only. Why Does It Look Like a Kids’ Toy?
Meta says Muse is just for adults, though its cuddly, Labubu-like mascot—and upcoming Tamagotchi-style AI device—may be disarming for users of

In [8]:
# Define nuestro mensaje de sistema: puedes experimentar con esto más tarde, cambiando la última oración a "Responder en Markdown en español".

system_prompt = "Eres un asistente que analiza el contenido de un sitio web \
y proporciona un breve resumen, ignorando el texto que podría estar relacionado con la navegación. \
Responder en Mark con referencias muy geek y d eingeniero humor tipo rick and morty big bang theory etcdown."

In [9]:
def user_prompt_for(website):
    user_prompt = f"Estás viendo un sitio web titulado {website.title}"
    user_prompt += "\nEl contenido de este sitio web es el siguiente; \
    proporciona un breve resumen de este sitio web en formato Markdown. \
    Si incluye noticias, productos o anuncios, resúmelos también.\n\n"
    user_prompt += website.text
    return user_prompt

In [11]:
print(user_prompt_for(pagina))

Estás viendo un sitio web titulado WIRED - The Latest in Technology, Science, Culture and Business | WIRED
El contenido de este sitio web es el siguiente;     proporciona un breve resumen de este sitio web en formato Markdown.     Si incluye noticias, productos o anuncios, resúmelos también.

Skip to main content
THE WIRED APP IS HERE
DOWNLOAD NOW »
Menu
WIRED
SECURITY
POLITICS
THE BIG STORY
BUSINESS
SCIENCE
CULTURE
REVIEWS
Menu
WIRED
Account
Account
Newsletters
Security
Politics
The Big Story
Business
Science
Culture
Reviews
Chevron
More
Expand
The Big Interview
Magazine
Events
WIRED Insider
WIRED Consulting
Newsletters
Podcasts
Video
Livestreams
WIRED Store
Search
Search
WIRED
Today’s Picks
Curating...
Headphone
Listen
Power Play
The Data Center Backlash Should Also Be a Climate Reckoning. It Isn’t Yet
Climate groups are latching onto the data center opposition movement, but it remains to be seen if they can get people to care about the facilities’ impact on the atmosphere.
By
Molly 

In [12]:
# Puedes ver cómo esta función crea exactamente el formato anterior

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

In [15]:
messages_for(pagina)

[{'role': 'system',
  'content': 'Eres un asistente que analiza el contenido de un sitio web y proporciona un breve resumen, ignorando el texto que podría estar relacionado con la navegación. Responder en Mark con referencias muy geek y d eingeniero humor tipo rick and morty big bang theory etcdown.'},
 {'role': 'user',
  'content': "Estás viendo un sitio web titulado WIRED - The Latest in Technology, Science, Culture and Business | WIRED\nEl contenido de este sitio web es el siguiente;     proporciona un breve resumen de este sitio web en formato Markdown.     Si incluye noticias, productos o anuncios, resúmelos también.\n\nSkip to main content\nTHE WIRED APP IS HERE\nDOWNLOAD NOW »\nMenu\nWIRED\nSECURITY\nPOLITICS\nTHE BIG STORY\nBUSINESS\nSCIENCE\nCULTURE\nREVIEWS\nMenu\nWIRED\nAccount\nAccount\nNewsletters\nSecurity\nPolitics\nThe Big Story\nBusiness\nScience\nCulture\nReviews\nChevron\nMore\nExpand\nThe Big Interview\nMagazine\nEvents\nWIRED Insider\nWIRED Consulting\nNewsletters\

In [16]:
# Y ahora: llama a la API de OpenAI. ¡Te resultará muy familiar!

def summarize(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model = "gpt-4o-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [17]:
summarize("https://wired.com")

'```markdown\n# Resumen de WIRED\n\n**WIRED** es el baluarte del conocimiento contemporáneo en tecnología, ciencia, cultura y negocios. Con un enfoque de ingeniería 2.0, el sitio se dedica a desenmarañar los hilos de lo que define nuestro presente y moldea nuestro futuro. Ya sabes, como si fuera un episodio de *Rick and Morty* donde cada decisión tiene un efecto mariposa en la realidad multiversal.\n\n## Principales Artículos\n- **Power Play:** El auge del movimiento contra los centros de datos como una cuestión climática que, irónicamente, todavía no capta la atención de todos.\n- **Fuzzy Logic:** Meta lanza su nuevo juguete, Muse, que podría parecer un Tamagotchi para adultos... porque, claro, si no eres un niño, ¿quién necesita juguetes adorables?\n- **Kernel Panic:** Los viejos trucos de estafa de tarjetas de crédito resurgen mientras el mundo se enfrenta a amenazas de IA. Es como si los fraudes tuviesen su propia temporada *retro*.\n\n## Resumen de Noticias\n- **Ciberseguridad:** 